In [1]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv('/Users/nrcase/CSC522/CSC522-Project/dataset_with_labels.csv')
df.head()

#print(df[df["name"] == "APT."])

# print(df['popularity'].value_counts())
# df['popularity'].value_counts().plot(kind='bar')
# plt.title('Popularity Distribution')

,spotify_id,name,artists,daily_rank,daily_movement,weekly_movement,country,snapshot_date,popularity,is_explicit,...,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,average_song
0,2plbrEY59IikOBgBGLjaoe,Die With A Smile,"Lady Gaga, Bruno Mars",1,1,0,NaN,2025-02-17,98,False,...,-7.777,0,0.0304,0.3080,0.0000,0.122,0.535,157.969,3,Lower
1,2CGNAOSuO1MEFCbBRgUzjd,luther (with sza),"Kendrick Lamar, SZA",2,1,4,NaN,2025-02-17,90,False,...,-7.546,1,0.1250,0.2510,0.0000,0.248,0.576,138.008,4,About_Average
2,6AI3ezQ4o3HUoP6Dhudph3,Not Like Us,Kendrick Lamar,3,-2,8,NaN,2025-02-17,92,True,...,-7.001,1,0.0776,0.0107,0.0000,0.141,0.214,101.061,4,Higher
3,4wJ5Qq0jBN4ajy7ouZIV1c,APT.,"ROSÉ, Bruno Mars",4,0,-2,NaN,2025-02-17,89,False,...,-4.477,0,0.2600,0.0283,0.0000,0.355,0.939,149.027,4,Higher
4,6dOtVTDdiauQNBQEDOtlAB,BIRDS OF A FEATHER,Billie Eilish,5,1,-2,NaN,2025-02-17,96,False,...,-10.171,1,0.0358,0.2000,0.0608,0.117,0.438,104.978,4,About_Average


In [2]:

from sklearn.preprocessing import MinMaxScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import HistGradientBoostingRegressor

#a = df[df["name"] == "APT."]
X = df.drop(columns=["spotify_id", "name", "artists", "snapshot_date", "country", "album_name", "album_release_date"], axis=1, inplace=False)
y = df["popularity"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(X_train.columns)

scaler = MinMaxScaler()
encode = OneHotEncoder()


Index(['daily_rank', 'daily_movement', 'weekly_movement', 'popularity',
       'is_explicit', 'duration_ms', 'danceability', 'energy', 'key',
       'loudness', 'mode', 'speechiness', 'acousticness', 'instrumentalness',
       'liveness', 'valence', 'tempo', 'time_signature', 'average_song'],
      dtype='object')


In [6]:
from sklearn.compose import make_column_transformer
from sklearn.metrics import mean_squared_error
from sklearn.metrics import root_mean_squared_error


preprocessing = make_column_transformer((encode, ['average_song']), (scaler, ['key', 'daily_rank', 'daily_movement', 'weekly_movement', 'popularity',
       'is_explicit', 'duration_ms', 'danceability', 'energy', 'key',
       'loudness', 'mode', 'speechiness', 'acousticness', 'instrumentalness',
       'liveness', 'valence', 'tempo', 'time_signature'] ), remainder='passthrough')

pipeline = make_pipeline(
    preprocessing,
    HistGradientBoostingRegressor()
)
pipeline.fit(X_train, y_train)
y_pred = pipeline.predict(X_test)

print("MSE: ", mean_squared_error(y_test, y_pred))
print("RMSE: ", root_mean_squared_error(y_test, y_pred))


MSE:  2.319601375092615e-07
RMSE:  0.0004816224013781559


In [4]:
pred = pd.DataFrame(y_pred).value_counts()
test = pd.DataFrame(y_test).value_counts()

print(pred.describe())
print(test.describe())

count      101.000000
mean      3422.544554
std       3623.056051
min         16.000000
25%         84.000000
50%       1794.000000
75%       5767.000000
max      12416.000000
Name: count, dtype: float64
count      101.000000
mean      3422.544554
std       3623.056051
min         16.000000
25%         84.000000
50%       1794.000000
75%       5767.000000
max      12416.000000
Name: count, dtype: float64
